# Taller: Detección de Casas Colombianas con YOLOv8
**Aplicaciones de Aprendizaje Automático de Máquinas**

Este notebook cubre:
1. Exploración del dataset
2. Fine-tuning de YOLOv8
3. Inferencia sobre imágenes nuevas
4. Visualización de resultados y métricas del modelo

## 0. Instalación de dependencias

In [ ]:
# Instalar Ultralytics si no está instalado
# !pip install ultralytics opencv-python matplotlib pyyaml
import ultralytics
ultralytics.checks()

## 1. Exploración del Dataset

In [ ]:
from pathlib import Path
import yaml
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import numpy as np

# Rutas
BASE_DIR   = Path(".").resolve()
DATA_YAML  = BASE_DIR / "casas.v1i.yolov8" / "data.yaml"
MODELS_DIR = BASE_DIR / "models"

# Leer configuración del dataset
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

print("=" * 50)
print("CONFIGURACIÓN DEL DATASET")
print("=" * 50)
print(f"Clases ({cfg['nc']}): {cfg['names']}")
print(f"Train : {cfg['train']}")
print(f"Val   : {cfg['val']}")
print(f"Test  : {cfg['test']}")

# Contar imágenes por split
dataset_root = Path(cfg['path'])
for split in ['train', 'valid', 'test']:
    imgs = list((dataset_root / split / 'images').glob('*'))
    lbls = list((dataset_root / split / 'labels').glob('*.txt'))
    print(f"  {split:6s}: {len(imgs)} imágenes, {len(lbls)} labels")

In [ ]:
# Visualizar imágenes del dataset con sus anotaciones (bounding boxes)
def mostrar_imagen_con_bbox(img_path: Path, label_path: Path, ax, titulo=""):
    """Muestra una imagen con sus bounding boxes anotados."""
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    ax.set_title(titulo, fontsize=9)
    ax.axis('off')

    if label_path.exists():
        with open(label_path) as f:
            for line in f:
                cls, cx, cy, bw, bh = map(float, line.strip().split())
                x1 = (cx - bw / 2) * w
                y1 = (cy - bh / 2) * h
                rect = patches.Rectangle(
                    (x1, y1), bw * w, bh * h,
                    linewidth=2, edgecolor='#00C864', facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(x1, y1 - 5, f"casa ({cls:.0f})",
                        color='#00C864', fontsize=8, fontweight='bold')

dataset_root = Path(cfg['path'])
train_imgs = sorted((dataset_root / 'train' / 'images').glob('*'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Dataset — Imágenes de entrenamiento con anotaciones YOLO', fontsize=13, fontweight='bold')

for ax, img_path in zip(axes.flat, train_imgs):
    lbl_path = dataset_root / 'train' / 'labels' / (img_path.stem + '.txt')
    mostrar_imagen_con_bbox(img_path, lbl_path, ax, titulo=img_path.stem[:30])

plt.tight_layout()
plt.savefig('resultados/dataset_anotaciones.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Guardado: resultados/dataset_anotaciones.png")

## 2. Fine-Tuning con YOLOv8

In [ ]:
from ultralytics import YOLO

# Cargar modelo base preentrenado en COCO (transfer learning)
model = YOLO('yolov8n.pt')
print(f"✅ Modelo base cargado: yolov8n.pt")
print(f"   Parámetros: {sum(p.numel() for p in model.model.parameters()):,}")

In [ ]:
# Fine-tuning
results = model.train(
    data    = str(DATA_YAML),
    epochs  = 50,
    imgsz   = 640,
    batch   = 8,
    device  = 0,        # GPU (cambiar a 'cpu' si no hay GPU)
    workers = 2,
    cache   = False,
    project = str(MODELS_DIR),
    name    = 'house-yolo',
    exist_ok= True,
)

print("\n✅ Entrenamiento completado")
print(f"   Resultados en: {results.save_dir}")

In [ ]:
# Copiar best.pt a models/ para entrega
import shutil

best_src  = Path(results.save_dir) / 'weights' / 'best.pt'
best_dest = MODELS_DIR / 'best.pt'
MODELS_DIR.mkdir(exist_ok=True)

if best_src.exists():
    shutil.copy(best_src, best_dest)
    print(f"✅ best.pt copiado a: {best_dest}")
else:
    print(f"⚠️ No se encontró best.pt en {best_src}")

## 3. Visualización de Métricas del Entrenamiento

In [ ]:
import pandas as pd

# Leer el CSV de resultados generado por Ultralytics
csv_path = Path(results.save_dir) / 'results.csv'
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()  # limpiar espacios

print("Columnas disponibles:", df.columns.tolist())
df.tail(5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Curvas de entrenamiento — Fine-Tuning YOLOv8n', fontsize=14, fontweight='bold')

metricas = [
    ('train/box_loss',           'Box Loss (train)',   '#E05252'),
    ('train/cls_loss',           'Class Loss (train)', '#E08C52'),
    ('train/dfl_loss',           'DFL Loss (train)',   '#E0C452'),
    ('metrics/mAP50(B)',         'mAP@0.5',            '#52A8E0'),
    ('metrics/precision(B)',     'Precision',          '#52E0A8'),
    ('metrics/recall(B)',        'Recall',             '#A852E0'),
]

for ax, (col, titulo, color) in zip(axes.flat, metricas):
    if col in df.columns:
        ax.plot(df['epoch'], df[col], color=color, linewidth=2)
        ax.fill_between(df['epoch'], df[col], alpha=0.15, color=color)
        ax.set_title(titulo, fontweight='bold')
        ax.set_xlabel('Época')
        ax.grid(True, alpha=0.3)
        ax.spines[['top', 'right']].set_visible(False)
    else:
        ax.set_title(f"{titulo} (no disponible)")
        ax.axis('off')

plt.tight_layout()
plt.savefig('resultados/curvas_entrenamiento.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Guardado: resultados/curvas_entrenamiento.png")

In [ ]:
# Tabla de métricas finales
ultima = df.iloc[-1]
print("=" * 40)
print("MÉTRICAS FINALES (época 50)")
print("=" * 40)
for col in ['metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/precision(B)', 'metrics/recall(B)']:
    if col in df.columns:
        nombre = col.split('/')[-1].replace('(B)', '').replace('metrics/', '')
        print(f"  {nombre:20s}: {ultima[col]:.4f}")

## 4. Inferencia sobre Imágenes Nuevas

In [ ]:
# Cargar el mejor modelo entrenado
best_model = YOLO(str(MODELS_DIR / 'best.pt'))
print(f"✅ Modelo cargado: best.pt")

# Imágenes a usar para inferencia (validación + test)
dataset_root = Path(cfg['path'])
imagenes_inf = (
    list((dataset_root / 'valid' / 'images').glob('*')) +
    list((dataset_root / 'test'  / 'images').glob('*'))
)
print(f"   Imágenes para inferencia: {len(imagenes_inf)}")

In [ ]:
# Correr inferencia
resultados_inf = best_model.predict(
    source  = [str(p) for p in imagenes_inf],
    conf    = 0.10,    # umbral bajo para ver más detecciones
    iou     = 0.45,
    save    = False,
    verbose = False,
)

print(f"✅ Inferencia completada sobre {len(resultados_inf)} imagen(es)")
for r in resultados_inf:
    print(f"  {Path(r.path).name}: {len(r.boxes)} detección(es)")

## 5. Visualización de Detecciones

In [ ]:
from pathlib import Path

Path('resultados').mkdir(exist_ok=True)

n_imgs = len(resultados_inf)
cols   = min(3, n_imgs)
rows   = (n_imgs + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
fig.suptitle('Inferencia — Detección de casas con YOLOv8n (fine-tuned)', fontsize=14, fontweight='bold')

if n_imgs == 1:
    axes = [axes]
else:
    axes = axes.flat

for ax, result in zip(axes, resultados_inf):
    img = result.orig_img.copy()
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)

    n_det = len(result.boxes)
    ax.set_title(f"{Path(result.path).name[:25]}\n{n_det} detección(es)", fontsize=9)
    ax.axis('off')

    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        conf = float(box.conf[0])
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor='#FF4444', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, max(y1 - 6, 0), f"casa {conf:.2f}",
                color='white', fontsize=8, fontweight='bold',
                bbox=dict(facecolor='#FF4444', alpha=0.7, pad=1, edgecolor='none'))

# Ocultar ejes sobrantes
for ax in list(axes)[n_imgs:]:
    ax.axis('off')

plt.tight_layout()
plt.savefig('resultados/inferencia_detecciones.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Guardado: resultados/inferencia_detecciones.png")

## 6. Visualización del Comportamiento Interno del Modelo

In [ ]:
# Visualización de la imagen de validación generada por Ultralytics
# (confusion matrix, PR curve, F1 curve)
run_dir = Path(results.save_dir)

graficas = [
    ('confusion_matrix.png', 'Matriz de Confusión'),
    ('PR_curve.png',         'Curva Precisión-Recall'),
    ('F1_curve.png',         'Curva F1'),
    ('results.png',          'Resumen del entrenamiento'),
]

disponibles = [(run_dir / f, t) for f, t in graficas if (run_dir / f).exists()]
print(f"Gráficas disponibles en {run_dir}:")
for p, t in disponibles:
    print(f"  - {p.name}")

if disponibles:
    cols = min(2, len(disponibles))
    rows = (len(disponibles) + 1) // 2
    fig, axes = plt.subplots(rows, cols, figsize=(12 * cols // 2, 6 * rows))
    fig.suptitle('Análisis del modelo — Ultralytics', fontsize=14, fontweight='bold')
    
    axes_flat = [axes] if len(disponibles) == 1 else axes.flat
    for ax, (path, titulo) in zip(axes_flat, disponibles):
        img = plt.imread(str(path))
        ax.imshow(img)
        ax.set_title(titulo, fontweight='bold')
        ax.axis('off')

    for ax in list(axes_flat)[len(disponibles):]:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('resultados/analisis_modelo.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("✅ Guardado: resultados/analisis_modelo.png")

In [ ]:
# Comparación lado a lado: imagen original vs imagen con detección
img_path = imagenes_inf[0]  # primera imagen
result   = resultados_inf[0]

img_orig = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
img_det  = result.orig_img.copy()
img_det  = cv2.cvtColor(img_det, cv2.COLOR_BGR2RGB)

# Dibujar boxes en img_det
for box in result.boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    conf = float(box.conf[0])
    cv2.rectangle(img_det, (x1, y1), (x2, y2), (255, 68, 68), 3)
    cv2.putText(img_det, f"casa {conf:.2f}", (x1, max(y1 - 8, 0)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 68, 68), 2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Comparación: Original vs Detección del modelo', fontsize=13, fontweight='bold')
ax1.imshow(img_orig); ax1.set_title('Imagen original', fontweight='bold'); ax1.axis('off')
ax2.imshow(img_det);  ax2.set_title(f'Detecciones YOLOv8n ({len(result.boxes)} casas)', fontweight='bold'); ax2.axis('off')
plt.tight_layout()
plt.savefig('resultados/comparacion_deteccion.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Guardado: resultados/comparacion_deteccion.png")

## 7. Resumen Final

In [ ]:
ultima = df.iloc[-1]

print("=" * 55)
print("  RESUMEN — TALLER YOLO DETECCIÓN DE CASAS")
print("=" * 55)
print(f"  Modelo base     : yolov8n.pt (transfer learning)")
print(f"  Dataset         : casas.v1i.yolov8 ({len(train_imgs)} train imgs)")
print(f"  Épocas          : 50")
print(f"  Imagen size     : 640x640")
print("")
print("  MÉTRICAS FINALES:")
for col in ['metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/precision(B)', 'metrics/recall(B)']:
    if col in df.columns:
        nombre = col.replace('metrics/', '').replace('(B)', '')
        print(f"    {nombre:20s}: {ultima[col]:.4f}")
print("")
print(f"  Pesos guardados : models/best.pt")
print("=" * 55)